# H1 check: overlapping vs. exclusive speaker diarization

Part of **video-rag**, a local pipeline that turns long-form interview videos
(the Spanish startup podcast by Itnig) into speaker-attributed transcripts and
then into chunks for a RAG system. Transcription and diarization run on
[WhisperX](https://github.com/m-bain/whisperX) +
[pyannote](https://github.com/pyannote/pyannote-audio).

## The problem

In video `yOLw6ncCJwY`, around 2:27, the host (Bernat) interjects *"¡Ostras! Esto
tiene mérito, ¿eh? ¿Y cómo...?"*, but the transcript attributes it to the guest
(Octavi). A wrong speaker there breaks the question/answer structure that the
later chunking relies on.

## Hypotheses

- **H1**: pyannote does detect Bernat, but as speech *overlapping* Octavi's.
  WhisperX assigns each word the speaker with the largest time overlap using
  pyannote's `speaker_diarization` output (which keeps overlaps), so Bernat's
  words tie with Octavi's and Octavi wins. If so, pyannote's
  `exclusive_speaker_diarization` (one speaker at a time) fixes it.
- **H2**: pyannote does not detect the speaker change at all. Then the exclusive
  output does not help either, and the fix has to come from somewhere else.

## What this notebook does

It downloads the audio, transcribes and aligns it once, diarizes it once, and
assigns word speakers with **both** pyannote outputs. For the 144-152 s window
it prints the raw pyannote intervals, every word with its speaker and the
resulting turns, so the two variants can be compared side by side.

It is self-contained (the project code is embedded, nothing is cloned) and runs
on Colab's own Python. Total time on a T4: about 15 minutes.

## Before running

1. *Runtime → Change runtime type → **T4 GPU***.
2. Create a Hugging Face token and add it in 🔑 *Secrets* as `HF_TOKEN`, with
   notebook access enabled.
3. With that same account, accept the terms of
   [`pyannote/speaker-diarization-community-1`](https://huggingface.co/pyannote/speaker-diarization-community-1)
   (the diarization model WhisperX 3.8.6 uses).

Then *Runtime → Run all*.

## 1. Install

In [ ]:
!nvidia-smi -L
!pip install -q whisperx==3.8.6 yt-dlp python-dotenv

## 2. Project code

The `video_rag` package, embedded as a compressed archive because the repository
is not cloned. The check uses its turn-building logic (`build_turns`), so the
turns printed below are exactly what the pipeline would produce.

In [ ]:
# @title Embedded video_rag package { display-mode: "form" }
import base64
import io
import os
import tarfile

WORKDIR = "/content/video-rag"
PACKAGE = (
    "..."
    "RAIPfw9/D38Pfw9/5u//B5zvf08AkAEA"
)
os.makedirs(WORKDIR, exist_ok=True)
archive = io.BytesIO(base64.b64decode(PACKAGE))
with tarfile.open(fileobj=archive, mode="r:gz") as tar:
    tar.extractall(WORKDIR, filter="data")
%cd {WORKDIR}

## 3. Configuration

The Hugging Face token goes into the environment, where the project reads it.

WhisperX also needs NLTK's `punkt_tab` sentence tokenizer and downloads it on
first use, but recent NLTK versions refuse any download through a proxy (an SSRF
protection), and Colab goes through one. It is fetched here instead.

In [ ]:
import urllib.request
import zipfile
from pathlib import Path

from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

NLTK_DATA = Path(WORKDIR) / "nltk_data"
PUNKT_TAB = (
    "https://raw.githubusercontent.com/nltk/nltk_data/gh-pages/packages/tokenizers/punkt_tab.zip"
)
tokenizers = NLTK_DATA / "tokenizers"
if not (tokenizers / "punkt_tab").is_dir():
    with urllib.request.urlopen(PUNKT_TAB) as response:
        zipfile.ZipFile(io.BytesIO(response.read())).extractall(tokenizers)
os.environ["NLTK_DATA"] = str(NLTK_DATA)

## 4. Audio

The whole episode (47 min), not a clip: diarization with only a minute of
context clusters voices much worse, and the error showed up on the full video.

If YouTube blocks the download from Colab (*"Sign in to confirm you're not a
bot"*), upload `yOLw6ncCJwY.wav` (16 kHz mono) to `/content/video-rag/audio/`
and run the next cells.

In [ ]:
import yt_dlp

VIDEO_ID = "yOLw6ncCJwY"
AUDIO = f"audio/{VIDEO_ID}.wav"

if not Path(AUDIO).exists():
    opts = {
        "format": "bestaudio/best",
        "outtmpl": "audio/%(id)s.%(ext)s",
        "postprocessors": [{"key": "FFmpegExtractAudio", "preferredcodec": "wav"}],
        "postprocessor_args": ["-ar", "16000", "-ac", "1"],
        "quiet": True,
    }
    with yt_dlp.YoutubeDL(opts) as ydl:
        ydl.download([f"https://www.youtube.com/watch?v={VIDEO_ID}"])
print(AUDIO, Path(AUDIO).stat().st_size // 1_000_000, "MB")

## 5. The check

`notebooks/h1_check.py` from the repository, written to disk as is. The window
144-152 s covers *"Sí. Y con una niña de dos años. ¡Ostras! Esto tiene mérito,
¿eh? ¿Y cómo...?"*. It takes about 10 minutes: `large-v2` transcription,
alignment and diarization of the full episode.

In [ ]:
%%writefile h1_check.py
"""H1 check: does pyannote's exclusive diarization fix speaker attribution?

Transcribes and aligns once, diarizes once, then assigns word speakers with
both pyannote outputs (`speaker_diarization`, the one WhisperX uses, and
`exclusive_speaker_diarization`) and prints, for the [start, end] window:
the raw pyannote intervals, the words with their speaker and the resulting
turns. Both full transcripts are saved to output/ for a wider comparison.

    PYTHONPATH=. python notebooks/h1_check.py audio/yOLw6ncCJwY.wav 144 152

It is embedded in the Colab notebook built by build_h1_notebook.py.
"""

import copy
import json
import sys
from pathlib import Path

import pandas as pd
import torch
import whisperx
from whisperx.diarize import DiarizationPipeline

from video_rag.config import Settings
from video_rag.domain.turns import build_turns
from video_rag.infrastructure.transcription.whisperx_recognizer import _segment


def to_df(annotation) -> pd.DataFrame:
    df = pd.DataFrame(
        annotation.itertracks(yield_label=True), columns=["segment", "label", "speaker"]
    )
    df["start"] = df["segment"].apply(lambda s: s.start)
    df["end"] = df["segment"].apply(lambda s: s.end)
    return df


def in_window(start, end, t0, t1) -> bool:
    return start is not None and end is not None and end >= t0 and start <= t1


def main() -> None:
    audio_path, t0, t1 = sys.argv[1], float(sys.argv[2]), float(sys.argv[3])
    hf_token = Settings.from_env().hf_token
    if not hf_token:
        sys.exit("HF_TOKEN is not set.")

    device = "cuda" if torch.cuda.is_available() else "cpu"
    compute_type = "float16" if device == "cuda" else "int8"
    audio = whisperx.load_audio(audio_path)

    print(f"Transcribing on {device} ...", flush=True)
    model = whisperx.load_model("large-v2", device, compute_type=compute_type, language="es")
    result = model.transcribe(audio, batch_size=16, language="es")
    del model
    print("Aligning ...", flush=True)
    model_a, metadata = whisperx.load_align_model(language_code="es", device=device)
    aligned = whisperx.align(
        result["segments"], model_a, metadata, audio, device, return_char_alignments=False
    )
    del model_a
    if device == "cuda":
        torch.cuda.empty_cache()

    print("Diarizing ...", flush=True)
    pipeline = DiarizationPipeline(token=hf_token, device=device)
    output = pipeline.model(
        {"waveform": torch.from_numpy(audio[None, :]), "sample_rate": 16000}, min_speakers=2
    )

    stem = Path(audio_path).stem
    variants = {
        "overlapping": output.speaker_diarization,
        "exclusive": output.exclusive_speaker_diarization,
    }
    for name, annotation in variants.items():
        df = to_df(annotation)
        print(f"\n=== {name}: pyannote intervals in [{t0}, {t1}] ===")
        for _, row in df.iterrows():
            if in_window(row["start"], row["end"], t0, t1):
                print(f"  {row['start']:8.3f} - {row['end']:8.3f}  {row['speaker']}")

        assigned = whisperx.assign_word_speakers(df, copy.deepcopy(aligned))
        segments = [_segment(s) for s in assigned["segments"]]

        print(f"--- {name}: words ---")
        for seg in segments:
            for w in seg.words:
                if in_window(w.start, w.end, t0, t1):
                    print(f"  {w.start:8.3f} {w.speaker}  {w.text}")

        turns = [t.rounded() for t in build_turns(segments, 4)]
        print(f"--- {name}: turns ({len(turns)} in the whole video) ---")
        for t in turns:
            if in_window(t.start, t.end, t0, t1):
                print(f"  {t.start:8.3f}-{t.end:8.3f} {t.speaker}: {t.text}")

        out = Path("output") / f"{stem}.{name}.json"
        out.parent.mkdir(exist_ok=True)
        out.write_text(
            json.dumps([t.to_dict() for t in turns], indent=2, ensure_ascii=False) + "\n",
            encoding="utf-8",
        )
        print(f"  -> {out}")


if __name__ == "__main__":
    main()

In [ ]:
T0, T1 = 144, 152
!python h1_check.py {AUDIO} {T0} {T1}

## 6. Summary and downloads

In [ ]:
import json

from google.colab import files

for name in ("overlapping", "exclusive"):
    path = f"output/{VIDEO_ID}.{name}.json"
    turns = json.loads(Path(path).read_text(encoding="utf-8"))
    speakers = sorted({t["speaker"] for t in turns})
    print(f"{name:12} {len(turns):4} turns  {', '.join(speakers)}")
    files.download(path)

## How to read the results

`SPEAKER_xx` labels are numbered by pyannote on every run, so check which one is
Bernat first: he is the one saying *"¿Y es lo que hiciste?"* just before the
window.

- **H1 confirmed**: in `overlapping` there is a Bernat interval overlapping one
  of Octavi's around 147-150 s, and in `exclusive` *"¡Ostras! Esto tiene mérito,
  ¿eh?"* carries Bernat's label and comes out as a turn of its own.
- **H2**: in `overlapping` there is no Bernat interval in that window at all.

If H1 holds, the fix in the pipeline is to assign word speakers from
`exclusive_speaker_diarization` instead of `speaker_diarization`.